In [15]:
import pandas as pd
import numpy as np
import os

In [2]:
train = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/Hotel Chain A - Supporting Resources/Hotel-A-train.csv')
val   = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/Hotel Chain A - Supporting Resources/Hotel-A-validation.csv')
test  = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/Hotel Chain A - Supporting Resources/Hotel-A-test.csv')

In [3]:
print('Train shape     :', train.shape)
print('Validation shape:', val.shape)
print('Test shape      :', test.shape)

Train shape     : (27499, 24)
Validation shape: (2749, 24)
Test shape      : (4318, 23)


## Missing values

In [4]:
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    print(f'\n{name}:')
    missing = df.isnull().sum()
    pct = (missing / len(df)*100.0).round(2)
    summary = pd.DataFrame({'Missing': missing, 'Pct(%)': pct})
    print(summary[summary['Missing'] > 0].to_string() or 'No missing values')


TRAIN:
Empty DataFrame
Columns: [Missing, Pct(%)]
Index: []

VALIDATION:
Empty DataFrame
Columns: [Missing, Pct(%)]
Index: []

TEST:
Empty DataFrame
Columns: [Missing, Pct(%)]
Index: []


## Check Duplicates

In [6]:
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    print(f'{name} duplicates: {df.duplicated().sum()}')

TRAIN duplicates: 0
VALIDATION duplicates: 0
TEST duplicates: 0


## Convert Date Columns

In [7]:
date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']

for df in [train, val, test]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Verify
print(train[date_cols].dtypes)

Expected_checkin     datetime64[ns]
Expected_checkout    datetime64[ns]
Booking_date         datetime64[ns]
dtype: object


## Fix Reservation_status casing

In [9]:
# Before fix
print('BEFORE FIX:')
print('TRAIN:', train['Reservation_Status'].value_counts())
print('VAL  :', val['Reservation_Status'].value_counts())

# Apply fix
train['Reservation_Status'] = (train['Reservation_Status']
                                .str.strip()
                                .replace({'Check-out': 'Check-Out',
                                          'Canceled' : 'Cancelled'}))

val['Reservation_Status'] = (val['Reservation_Status']
                              .str.strip()
                              .replace({'Check-In' : 'Check-Out',
                                        'Canceled' : 'Cancelled'}))

# After fix
print('\nAFTER FIX:')
print('TRAIN:', train['Reservation_Status'].value_counts())
print('VAL  :', val['Reservation_Status'].value_counts())

BEFORE FIX:
TRAIN: Reservation_Status
Check-Out    21240
Canceled      4134
No-Show       2125
Name: count, dtype: int64
VAL  : Reservation_Status
Check-Out    1610
Canceled      741
No-Show       398
Name: count, dtype: int64

AFTER FIX:
TRAIN: Reservation_Status
Check-Out    21240
Cancelled     4134
No-Show       2125
Name: count, dtype: int64
VAL  : Reservation_Status
Check-Out    1610
Cancelled     741
No-Show       398
Name: count, dtype: int64


## Encode Reservation_Status numerically

In [10]:
status_map = {'Check-Out': 1, 'Cancelled': 2, 'No-Show': 3}

train['Reservation_Status_Code'] = train['Reservation_Status'].map(status_map)
val['Reservation_Status_Code']   = val['Reservation_Status'].map(status_map)

# Verify — should show no NaN
print('TRAIN NaN in status code:', train['Reservation_Status_Code'].isnull().sum())
print('VAL   NaN in status code:', val['Reservation_Status_Code'].isnull().sum())

TRAIN NaN in status code: 0
VAL   NaN in status code: 0


## Create binary target Variables

In [11]:
train['Cancelled'] = (train['Reservation_Status'] == 'Cancelled').astype(int)
train['No_Show']   = (train['Reservation_Status'] == 'No-Show').astype(int)
val['Cancelled']   = (val['Reservation_Status'] == 'Cancelled').astype(int)
val['No_Show']     = (val['Reservation_Status'] == 'No-Show').astype(int)

# Verify
print('TRAIN targets:')
print(train.groupby('Reservation_Status')[['Cancelled', 'No_Show']].sum())

print('\nVAL targets:')
print(val.groupby('Reservation_Status')[['Cancelled', 'No_Show']].sum())

TRAIN targets:
                    Cancelled  No_Show
Reservation_Status                    
Cancelled                4134        0
Check-Out                   0        0
No-Show                     0     2125

VAL targets:
                    Cancelled  No_Show
Reservation_Status                    
Cancelled                 741        0
Check-Out                   0        0
No-Show                     0      398


## Check and fix capacity violations

In [12]:
# Check before fix
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    violations = df[df['Adults'] + df['Children'] > 5]
    print(f'{name} — violations before fix: {len(violations)}')

# Apply fix — cap Children so Adults + Children never exceeds 5
for df in [train, val, test]:
    df['Children'] = df.apply(lambda row: min(row['Children'], 5 - row['Adults']), axis=1)

# Verify
print()
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    violations = df[df['Adults'] + df['Children'] > 5]
    print(f'{name} — violations after fix: {len(violations)}')

TRAIN — violations before fix: 4369
VALIDATION — violations before fix: 422
TEST — violations before fix: 727

TRAIN — violations after fix: 0
VALIDATION — violations after fix: 0
TEST — violations after fix: 0


## Inconsistent data

In [17]:
# Age range Check
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    invalid_age = df[(df['Age'] < 18) | (df['Age'] > 70)]
    print(f'{name} — Age out of range (< 18 or > 70): {len(invalid_age)}')

TRAIN — Age out of range (< 18 or > 70): 0
VALIDATION — Age out of range (< 18 or > 70): 0
TEST — Age out of range (< 18 or > 70): 0


In [20]:
# check negative lead times

date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']
for df in [train, val, test]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Check
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    negative_lead = df[df['Expected_checkin'] < df['Booking_date']]
    print(f'{name} — Negative lead time (check-in before booking): {len(negative_lead)}')

TRAIN — Negative lead time (check-in before booking): 506
VALIDATION — Negative lead time (check-in before booking): 16
TEST — Negative lead time (check-in before booking): 27


In [22]:
# Invalide stay duration (check-out before check-in)

for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    invalid_stay = df[df['Expected_checkout'] <= df['Expected_checkin']]
    print(f'{name} — Invalid stay duration (checkout <= check-in): {len(invalid_stay)}')

TRAIN — Invalid stay duration (checkout <= check-in): 0
VALIDATION — Invalid stay duration (checkout <= check-in): 0
TEST — Invalid stay duration (checkout <= check-in): 0


# Outliers

In [23]:
num_cols = ['Age', 'Adults', 'Children', 'Babies', 'Discount_Rate', 'Room_Rate']

print('Outlier counts per column (Train):')
for col in num_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = train[(train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)]
    print(f'  {col:<20} — outliers: {len(outliers)}')

Outlier counts per column (Train):
  Age                  — outliers: 0
  Adults               — outliers: 2286
  Children             — outliers: 0
  Babies               — outliers: 0
  Discount_Rate        — outliers: 0
  Room_Rate            — outliers: 0


In [24]:
# Fix — cap outliers using IQR bounds (Winsorization)
for df in [train, val, test]:
    for col in num_cols:
        Q1  = train[col].quantile(0.25)
        Q3  = train[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower, upper=upper)

# Verify
print('Outlier counts after fix (Train):')
for col in num_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = train[(train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)]
    print(f'  {col:<20} — outliers: {len(outliers)}')

Outlier counts after fix (Train):
  Age                  — outliers: 0
  Adults               — outliers: 0
  Children             — outliers: 0
  Babies               — outliers: 0
  Discount_Rate        — outliers: 0
  Room_Rate            — outliers: 0


In [25]:
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    print(f'\n{name}')
    print(f'  Shape          : {df.shape}')
    print(f'  Missing values : {df.isnull().sum().sum()}')
    print(f'  Duplicates     : {df.duplicated().sum()}')


TRAIN
  Shape          : (27499, 27)
  Missing values : 0
  Duplicates     : 0

VALIDATION
  Shape          : (2749, 27)
  Missing values : 0
  Duplicates     : 0

TEST
  Shape          : (4318, 23)
  Missing values : 0
  Duplicates     : 0


In [26]:
#Create folder
os.makedirs('cleaned_dataset', exist_ok=True)

# Save cleaned datasets
train.to_csv('cleaned_dataset/train_cleaned.csv', index=False)
val.to_csv('cleaned_dataset/val_cleaned.csv',     index=False)
test.to_csv('cleaned_dataset/test_cleaned.csv',   index=False)

print('Cleaned datasets saved to cleaned_dataset folder.')

Cleaned datasets saved to cleaned_dataset folder.
